# Vanilla Transformer — Chinese ↔ Vietnamese baseline (Colab)

Notebook chạy baseline **Transformer 30.8M** (kiến trúc encoder-decoder cổ điển) trên
**cùng data + cùng tokenizer + cùng training loop** với Bi-Mamba 32.4M, để
trả lời câu hỏi: *BLEU ~20 hiện tại bị giới hạn bởi kiến trúc Mamba hay bởi data?*

* Nếu Transformer đạt **>25 BLEU** trên cùng test set → vấn đề là kiến trúc Bi-Mamba seq2seq.
* Nếu Transformer cũng **kẹt ~20 BLEU** → vấn đề là data/tokenizer/preprocessing.

Notebook này **không thay đổi gì trong `bi_mamba_mt/`**; mọi thứ shared
(tokenizer, data, evaluator, trainer) đến từ `mt_base/`.


## 1. Mount Google Drive (tuỳ chọn)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/bi_mamba_zh_vi'
    import os
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print('Mounted Drive at', DRIVE_DIR)
except Exception as e:
    print('Skip Drive (probably not running on Colab):', e)
    DRIVE_DIR = None


## 2. Clone repo

In [ ]:
import os
REPO_URL = 'https://github.com/ChauDucToan/NLP_DHM.git'
WORK_DIR = '/content/NLP_DHM'

if not os.path.isdir(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
%cd {WORK_DIR}
!git pull --ff-only


## 3. Cài đặt dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .


## 4. Cấu hình Colab

Dùng `configs/transformer_30m.yaml` (cùng `data:` và `tokenizer:` với Bi-Mamba — đảm bảo apples-to-apples).

In [ ]:
import os, yaml, shutil
os.makedirs('configs', exist_ok=True)
shutil.copy('configs/transformer_30m.yaml', 'configs/_colab.yaml')

with open('configs/_colab.yaml') as f:
    cfg = yaml.safe_load(f)

# T4 không hỗ trợ bf16 native → dùng fp16
cfg['train']['amp_dtype'] = 'fp16'
cfg['train']['num_workers'] = 4
cfg['train']['output_dir'] = 'runs/transformer_30m'

with open('configs/_colab.yaml', 'w') as f:
    yaml.dump(cfg, f, sort_keys=False, allow_unicode=True)

print('Config OK:')
print('  preset =', cfg['data']['preset'])
print('  d_model =', cfg['model']['d_model'])
print('  layers =', cfg['model']['n_encoder_layers'], '+', cfg['model']['n_decoder_layers'])
print('  d_ff =', cfg['model']['d_ff'])
print('  max_steps =', cfg['train']['max_steps'])
print('  output_dir =', cfg['train']['output_dir'])


## 5. Tải + chuẩn bị data (OPUS zh-vi, preset `everyday`)

Nếu đã chạy notebook Bi-Mamba trước thì skip — `data/processed/*.jsonl` và `data/tokenizer/spm.model` dùng chung.

In [ ]:
import os
PROC = 'data/processed'
if not (os.path.exists(f'{PROC}/train.jsonl') and os.path.exists('data/tokenizer/spm.model')):
    !python scripts/prepare_data.py --config configs/_colab.yaml --preset everyday
    !python scripts/train_tokenizer.py --config configs/_colab.yaml
else:
    print('Data + tokenizer đã có sẵn → skip.')
    !ls -lh data/processed/ data/tokenizer/


## 6. Khởi tạo model + kiểm tra số tham số

In [ ]:
from transformer_mt.model import TransformerTranslator, ModelConfig
from mt_base.utils import count_parameters, human_format
import yaml

with open('configs/_colab.yaml') as f:
    cfg = yaml.safe_load(f)

mcfg = ModelConfig(**cfg['model'])
m = TransformerTranslator(mcfg)
n = count_parameters(m)
print(f'Transformer params: {human_format(n)} ({n:,})')
print(f'  vs Bi-Mamba 32.43M target')


## 7. Train

In [ ]:
# rm -rf runs/transformer_30m  # uncomment để clean restart
!python scripts/train_transformer.py --config configs/_colab.yaml


## 8. Average last-5 + EMA checkpoints

In [ ]:
!python scripts/avg_ckpts.py --ckpts-dir runs/transformer_30m --n 5
!python scripts/avg_ckpts.py --ckpts-dir runs/transformer_30m --n 5 --ema
!ls -lh runs/transformer_30m/*.pt | head -20


## 9. Đánh giá SacreBLEU + chrF (cả hai chiều)

Per-length-bucket breakdown (`--length-buckets`) bật mặc định để so sánh trực tiếp với Bi-Mamba.

In [ ]:
# Compare checkpoint variants on the test set + per-length-bucket breakdown.
import os
ckpts = []
for name in ['best.pt', 'best_ema.pt', 'avg_last5.pt', 'avg_last5_ema.pt']:
    p = f'runs/transformer_30m/{name}'
    if os.path.exists(p):
        ckpts.append(p)

print('Eval candidates:', ckpts)
for ckpt in ckpts:
    print(f'\n=== {ckpt} (beam=4, default LP) ===')
    !python scripts/evaluate_transformer.py --config configs/_colab.yaml \
        --checkpoint {ckpt} --num-samples 5000 --beam-size 4 --length-buckets


## 9b. Grid-sweep decoding (beam × length_penalty → CSV)

Tái dùng `scripts/sweep_decode.py` — nhưng script đó hardcode loader Bi-Mamba.
Để dùng cho Transformer, ta **eval thủ công lưới nhỏ** ở đây.

In [ ]:
import os, csv, itertools, subprocess, re

# Pick best available checkpoint
for cand in ['avg_last5_ema.pt', 'best_ema.pt', 'avg_last5.pt', 'best.pt']:
    p = f'runs/transformer_30m/{cand}'
    if os.path.exists(p):
        sweep_ckpt = p
        break

print(f'Sweep target: {sweep_ckpt}')
csv_path = 'runs/transformer_30m/sweep.csv'

beams = [1, 2, 4, 6]
lp_zh2vi = [0.8, 0.9, 1.0, 1.1, 1.2]
lp_vi2zh = [0.6, 0.8, 0.9, 1.0]

rows = []
def run_eval(direction, beam, lp):
    cmd = [
        'python', 'scripts/evaluate_transformer.py',
        '--config', 'configs/_colab.yaml',
        '--checkpoint', sweep_ckpt,
        '--num-samples', '2000',
        '--directions', direction,
        '--beam-size', str(beam),
        f'--length-penalty-{direction.replace("2", "2")}', str(lp),
    ]
    out = subprocess.run(cmd, capture_output=True, text=True)
    m = re.search(rf'\[{direction}\]\s+n=(\d+)\s+BLEU=([\d.]+)\s+chrF=([\d.]+)', out.stdout)
    if m:
        n, bleu, chrf = int(m.group(1)), float(m.group(2)), float(m.group(3))
        return n, bleu, chrf
    print('FAILED:', out.stderr[-500:])
    return 0, 0.0, 0.0

print('--- zh→vi sweep ---')
for beam, lp in itertools.product(beams, lp_zh2vi):
    n, bleu, chrf = run_eval('zh2vi', beam, lp)
    rows.append({'direction': 'zh2vi', 'beam': beam, 'lp': lp, 'n': n, 'bleu': bleu, 'chrf': chrf})
    print(f'  beam={beam} lp={lp:.2f} BLEU={bleu:.2f} chrF={chrf:.2f}')

print('--- vi→zh sweep ---')
for beam, lp in itertools.product(beams, lp_vi2zh):
    n, bleu, chrf = run_eval('vi2zh', beam, lp)
    rows.append({'direction': 'vi2zh', 'beam': beam, 'lp': lp, 'n': n, 'bleu': bleu, 'chrf': chrf})
    print(f'  beam={beam} lp={lp:.2f} BLEU={bleu:.2f} chrF={chrf:.2f}')

with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['direction', 'beam', 'lp', 'n', 'bleu', 'chrf'])
    w.writeheader()
    w.writerows(rows)
print(f'\nSaved {csv_path}')

# Top-3 per direction
import collections
by_dir = collections.defaultdict(list)
for r in rows:
    by_dir[r['direction']].append(r)
for d, rs in by_dir.items():
    rs.sort(key=lambda r: r['bleu'], reverse=True)
    print(f'\nTop-3 {d}:')
    for r in rs[:3]:
        print(f"  beam={r['beam']} lp={r['lp']:.2f} BLEU={r['bleu']:.2f} chrF={r['chrf']:.2f}")


## 10. Demo dịch (sanity check)

In [ ]:
import os
import torch
import yaml
from transformer_mt.model import TransformerTranslator, ModelConfig
from mt_base.tokenizer import Tokenizer
from mt_base.translate import translate_batch

with open('configs/_colab.yaml') as f:
    cfg = yaml.safe_load(f)

# Pick best available
for cand in ['avg_last5_ema.pt', 'best_ema.pt', 'avg_last5.pt', 'best.pt']:
    p = f'runs/transformer_30m/{cand}'
    if os.path.exists(p):
        ckpt_path = p
        break

print(f'Loading {ckpt_path}')
ckpt = torch.load(ckpt_path, map_location='cuda', weights_only=False)
mcfg = ModelConfig(**ckpt.get('model_cfg', cfg['model']))
m = TransformerTranslator(mcfg).cuda().eval()
m.load_state_dict(ckpt['model'], strict=False)

tok = Tokenizer('data/tokenizer/spm.model')

zh_samples = ['你好，世界。', '今天天气真好。', '这本书我已经读完了。']
vi_samples = ['Xin chào thế giới.', 'Hôm nay trời thật đẹp.', 'Tôi đã đọc xong cuốn sách này.']

print('--- zh → vi ---')
for s, t in zip(zh_samples, translate_batch(m, tok, zh_samples, 'zh2vi', beam_size=4, length_penalty=1.0)):
    print(f'  {s!r} -> {t!r}')

print('--- vi → zh ---')
for s, t in zip(vi_samples, translate_batch(m, tok, vi_samples, 'vi2zh', beam_size=4, length_penalty=0.9)):
    print(f'  {s!r} -> {t!r}')


## 11. (Tuỳ chọn) Lưu checkpoint vào Drive

In [ ]:
import shutil, os
if DRIVE_DIR:
    target = os.path.join(DRIVE_DIR, 'runs/transformer_30m')
    os.makedirs(target, exist_ok=True)
    for f in ['best.pt', 'best_ema.pt', 'avg_last5.pt', 'avg_last5_ema.pt', 'final.pt', 'config.yaml']:
        s = f'runs/transformer_30m/{f}'
        if os.path.exists(s):
            shutil.copy(s, target)
            print('Copied', f)
